# MBG Tweet Classification — XLM-RoBERTa-Large Pipeline

**Kompetisi:** BDC Internal 2026 — Case 1  
**Task:** Multiclass text classification (8 kelas) pada tweet berbahasa Indonesia tentang program Makan Bergizi Gratis (MBG)  
**Metrik evaluasi:** Balanced Accuracy  
**Arsitektur utama:** `xlm-roberta-large` dengan Multi-Sample Dropout dan 5-Fold Stratified Cross-Validation

---

## Ringkasan Metodologi

| Tahap | Keputusan | Alasan |
|---|---|---|
| Model backbone | XLM-RoBERTa-Large | Pretrained multilingual, superior pada teks informal Indonesia |
| Pooling | CLS token (`[s]`) | Standar RoBERTa; representasi seluruh sequence |
| Regularisasi | Multi-Sample Dropout (5 head) | Stabilisasi training, implicit ensemble pada satu model |
| Loss function | CrossEntropyLoss berbobot | Kompensasi distribusi kelas yang tidak seimbang |
| CV Strategy | 5-Fold Stratified K-Fold | Menjaga proporsi kelas di setiap fold |
| Optimasi | AdamW + Linear Warmup Scheduler | Standar fine-tuning transformer |
| Efisiensi GPU | AMP (Automatic Mixed Precision) + Gradient Accumulation | Fit model besar di VRAM terbatas |
| Preprocessing | Demojize + HTML unescape + slang normalization | Mengurangi noise UGC, membantu tokenizer |


---

## 1. Dependencies

Library yang digunakan mencakup:
- **PyTorch + HuggingFace Transformers** — backbone fine-tuning dan tokenisasi
- **scikit-learn** — stratified splitting, class weight computation, dan evaluasi
- **emoji** — demojizing emoji menjadi representasi teks yang dapat diproses tokenizer
- **tqdm** — progress monitoring per batch


In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import emoji
import warnings
warnings.filterwarnings('ignore')

---

## 2. Konfigurasi Eksperimen

### Pemilihan Model: `xlm-roberta-large`

XLM-RoBERTa-Large dipilih sebagai backbone karena dua alasan utama:

1. **Multilingual pretraining** — model ini dilatih pada 100 bahasa termasuk Bahasa Indonesia, sehingga representasi subword-nya sudah mencakup kosakata formal maupun informal yang umum di Twitter/X.
2. **Kapasitas model** — versi *large* (24 layer, 1024 hidden size) secara konsisten menghasilkan representasi yang lebih kaya dibandingkan *base* (12 layer, 768 hidden size) untuk tugas klasifikasi teks panjang dan bernuansa.

### Hyperparameter

| Parameter | Nilai | Justifikasi |
|---|---|---|
| `MAX_LEN` | 192 | Tweet jarang melebihi 280 karakter; 192 token mengcover >95% data dengan efisiensi VRAM |
| `BATCH_SIZE` | 8 | Batas aman untuk model Large di VRAM 16GB (P100/T4) |
| `GRAD_ACCUM_STEPS` | 2 | Effective batch size = 16; meniru perilaku batch besar tanpa overhead memori |
| `EPOCHS` | 3 | Fine-tuning transformer umumnya konvergen dalam 3-5 epoch; lebih dari itu berisiko overfitting |
| `LEARNING_RATE` | 1e-5 | LR yang lebih kecil dari base (2e-5) untuk menjaga bobot pretrained agar tidak terdegradasi |
| `N_SPLITS` | 5 | 5-fold memberi estimasi performa yang stabil dengan 80/20 train-val split per fold |
| `SEED` | 42 | Reproduktibilitas eksperimen |

> **Referensi:** Conneau, A., Khandelwal, K., Goyal, N., et al. (2020). *Unsupervised Cross-lingual Representation Learning at Scale*. Proceedings of ACL 2020. [arXiv:1911.02116](https://arxiv.org/abs/1911.02116)


In [ ]:
TRAIN_PATH = "/kaggle/input/datasets/yashanathaniel/satdat/case_1_labeled_data.xlsx - Sheet1.csv"
TEST_PATH = "/kaggle/input/datasets/yashanathaniel/satdat/case_1_text_to_predict.xlsx - Sheet1.csv"
SAMPLE_SUBMISSION_PATH = "/kaggle/input/datasets/yashanathaniel/satdat/case_1_template_sheet.xlsx - Sheet1.csv"
#  THE GOD TIER MODEL: XLM-RoBERTa-Large
# KARENA PATH KERAS GAGAL (Seperti yang diprediksi), kita KEMBALI menggunakan internet.
# ATAU jika harus offline, Anda WAJIB melampirkan dataset "xlm-roberta-large pytorch".
MODEL_NAME = "xlm-roberta-large"
MODELS_DIR = "/kaggle/working/models"

#  KAGGLE HYPERPARAMETERS (Asumsi VRAM 16GB - P100 / T4x2)
MAX_LEN = 192        # VRAM 16GB sanggup membaca konteks lebih panjang!
BATCH_SIZE = 8       # Model ini sangat raksasa, Batch 8 adalah batas aman di 16GB VRAM
GRAD_ACCUM_STEPS = 2 # Efektif Batch Size = 16 (8 x 2)
EPOCHS = 3           
LEARNING_RATE = 1e-5 # Learning rate RoBERTa-Large harus lebih kecil dari Base
N_SPLITS = 5
SEED = 42

def seed_everything(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(MODELS_DIR, exist_ok=True)
os.environ["TRANSFORMERS_CACHE"] = MODELS_DIR


---

## 3. Arsitektur Model: `XLMRobertaMultiDropout`

### CLS Token Pooling

Output yang diambil adalah `last_hidden_state[:, 0, :]` — yaitu representasi token `<s>` (CLS) pada layer terakhir. Pada arsitektur RoBERTa, token ini secara training dioptimalkan untuk merangkum informasi seluruh sequence, sehingga menjadi pilihan natural sebagai input ke layer klasifikasi.

### Multi-Sample Dropout

Alih-alih satu dropout layer, digunakan 5 dropout head paralel dengan rate 0.2. Setiap head menghasilkan prediksi logit yang kemudian dirata-ratakan sebelum dikembalikan.

Mekanisme ini bekerja seperti *implicit ensemble*: setiap forward pass mengekspos classifier ke versi yang sedikit berbeda dari representasi CLS, memaksa model untuk tidak bergantung pada neuron tertentu. Efeknya adalah:
- Konvergensi lebih cepat dan lebih stabil (efek regularisasi lebih kuat dari single dropout)
- Performa inference yang lebih robust karena averaging mengurangi varians prediksi

Pendekatan ini pertama kali diperkenalkan dalam konteks NLP oleh Inoue et al. dan kemudian dipopulerkan di kompetisi NLP Kaggle sebagai teknik regularisasi yang efektif untuk fine-tuning transformer.

Pendekatan ini pertama kali diperkenalkan oleh Inoue (2019) dan terbukti mempercepat konvergensi serta menurunkan error rate pada berbagai benchmark klasifikasi.

> **Referensi — Multi-Sample Dropout:** Inoue, H. (2019). *Multi-Sample Dropout for Accelerated Training and Better Generalization*. [arXiv:1905.09788](https://arxiv.org/abs/1905.09788)
>
> **Referensi — CLS token pooling (RoBERTa):** Liu, Y., Ott, M., Goyal, N., et al. (2019). *RoBERTa: A Robustly Optimized BERT Pretraining Approach*. [arXiv:1907.11692](https://arxiv.org/abs/1907.11692)


In [ ]:
class XLMRobertaMultiDropout(nn.Module):
    def __init__(self, model_name, num_labels):
        super(XLMRobertaMultiDropout, self).__init__()
        
        try:
            # Mencoba load normal (Jika folder memiliki format HuggingFace PyTorch)
            self.roberta = AutoModel.from_pretrained(model_name)
        except Exception as e:
            print(f"Load PyTorch gagal ({e}). Mencoba fallback ke format TensorFlow/Keras...")
            # Fallback jika model adalah TF/Keras
            self.roberta = AutoModel.from_pretrained(model_name, from_tf=True)
            
        self.config = self.roberta.config
        
        #  5 layer dropout paralel untuk stabilisasi model raksasa
        self.dropouts = nn.ModuleList([nn.Dropout(0.2) for _ in range(5)])
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        #  Mengambil token <s> (Indeks 0) sebagai penampung informasi seluruh kalimat di RoBERTa
        pooled_output = outputs.last_hidden_state[:, 0, :] 
        
        logits = 0
        for dropout in self.dropouts:
            x = dropout(pooled_output)
            logits += self.classifier(x)
            
        return logits / len(self.dropouts)


---

## 4. Preprocessing Teks

Data UGC dari Twitter memiliki karakteristik noise yang tidak ada dalam data pretraining standar. Langkah preprocessing dirancang untuk mengurangi noise tersebut secara targeted, bukan agresif — tujuannya membantu tokenizer, bukan menghapus sinyal semantik.

### Langkah-langkah dan Justifikasinya

**Step 0 — Demojize**  
Emoji dikonversi menjadi representasi teks deskriptif (contoh: `😡` → ` angry_face `). Alasannya: XLM-RoBERTa memiliki vocab terbatas untuk karakter Unicode emoji. Dengan demojize, makna emosional emoji tetap terjaga dalam bentuk teks yang dapat diproses tokenizer.

**Step 1 — HTML Unescape**  
Twitter API sering mengembalikan karakter HTML-encoded (`&amp;`, `&lt;`, dll.) akibat sanitasi pada backend mereka. Unescape memastikan teks yang dibaca model adalah teks asli pengguna.

**Step 2 — Kompresi Karakter Berulang**  
Pengguna Twitter sering menggunakan penekanan dengan pengulangan (`yaaaaaa`, `!!!!!!`). Kompresi ke satu karakter (`ya`, `!`) menormalkan variasi ini agar tidak menghasilkan token OOV yang tidak perlu.

**Step 3 — Normalisasi Slang Domain MBG**  
XLM-RoBERTa dilatih pada korpus formal. Singkatan dan slang yang spesifik pada diskusi MBG (`gk`, `tdk`, `bsi`, `bltg`) berpotensi menjadi token OOV atau mendapat representasi suboptimal. Kamus slang ini ditranslasikan ke bentuk baku agar model dapat menggunakan representasi pretrained yang sudah matang.

> **Catatan:** Preprocessing dijaga seminimal mungkin — tidak ada penghapusan stopword, stemming, atau lowercasing agresif — karena XLM-RoBERTa sudah menangani variasi kasus dan morfologi secara internal melalui subword tokenization (SentencePiece).


In [ ]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    import html
    import re
    
    # 0. Demojize: Mengubah emoji menjadi teks (Contoh: 😂 -> wajah_gembira)
    text = emoji.demojize(text, delimiters=(" ", " "))
    
    # 1. HTML Unescape (Membersihkan bug API: &amp; -> &)
    text = html.unescape(text)
    
    # 2. SURGICAL OBFUSCATION: Kompresi huruf & tanda baca berulang
    text = re.sub(r'([a-zA-Z])\1{2,}', r'\1', text)
    text = re.sub(r'([!?.])\1{2,}', r'\1', text)
    
    # 3. KAMUS SLANG DOMAIN MBG
    # XLM-RoBERTa adalah model multibahasa, ia jauh lebih presisi jika 
    # bahasa "slang" Indonesia diterjemahkan ke bahasa baku.
    domain_slang = {
        'yg': 'yang', 'gk': 'tidak', 'ga': 'tidak', 'nggak': 'tidak', 'tdk': 'tidak',
        'anggrn': 'anggaran', 'kmpny': 'kampanye', 'bltg': 'belatung',
        'bsi': 'basi', 'pmerntah': 'pemerintah', 'bnyk': 'banyak',
        'sdh': 'sudah', 'jgn': 'jangan', 'bgs': 'bagus', 'dpt': 'dapat'
    }
    words = text.split()
    words = [domain_slang.get(w.lower(), w) for w in words]
    text = ' '.join(words)
    
    return re.sub(r'\s+', ' ', text).strip()


---

## 5. Dataset dan Data Loading

### `TweetDataset`

Wrapper `torch.utils.data.Dataset` yang menghandle tokenisasi per-sample. Desain ini memisahkan logika tokenisasi dari logika training, sehingga lebih mudah di-debug dan dimodifikasi secara independen.

Padding strategy `max_length` dipilih (bukan dynamic padding) karena sederhana dan kompatibel dengan `DataLoader` standar. Trade-off: sedikit overhead komputasi untuk sampel pendek, namun eliminasi kebutuhan custom `collate_fn`.

### `load_data()`

`LabelEncoder` dari scikit-learn digunakan untuk mengkonversi label string menjadi integer. Penting: `fit` dilakukan hanya pada train set (`fit_transform`), sehingga mapping label konsisten dan dapat diinversikan untuk menghasilkan submission akhir.


In [ ]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, item):
        text = str(self.texts[item])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        sample = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        if self.labels is not None:
            sample['labels'] = torch.tensor(self.labels[item], dtype=torch.long)
        return sample

def load_data():
    df_train = pd.read_csv(TRAIN_PATH)
    df_test = pd.read_csv(TEST_PATH)
    
    df_train['clean_text'] = df_train['full_text'].apply(preprocess_text)
    df_test['clean_text'] = df_test['full_text'].apply(preprocess_text)
    
    le = LabelEncoder()
    df_train['label_encoded'] = le.fit_transform(df_train['label'])
    
    return df_train, df_test, le


---

## 6. Pipeline Training dan Inferensi

### Class-Weighted Loss

Dataset memiliki distribusi kelas yang tidak seimbang (diindikasikan oleh pilihan metrik *balanced accuracy* oleh panitia). `compute_class_weight('balanced')` dari scikit-learn menghitung bobot sebanding dengan invers frekuensi kelas, yang kemudian diinjeksikan ke `CrossEntropyLoss`. Efeknya: model mendapat penalti lebih besar jika salah memprediksi kelas minoritas, mendorong recall yang lebih merata di semua kelas.

### Stratified K-Fold (5 Fold)

`StratifiedKFold` memastikan proporsi setiap kelas terjaga di setiap fold train/val. Ini kritis pada data imbalanced — random split biasa bisa menghasilkan fold val yang tidak merepresentasikan kelas minoritas.

### AdamW + Linear Warmup Scheduler

Kombinasi standar untuk fine-tuning transformer:
- **AdamW** (Adam dengan weight decay decoupled) mencegah overfitting pada layer pretrained
- **Linear warmup** (10% dari total steps) memulai training dengan LR sangat kecil, memberi kesempatan model untuk "menyesuaikan diri" sebelum update penuh. Ini mencegah degradasi representasi pretrained di awal training
- **Linear decay** setelah warmup memastikan konvergensi halus menjelang akhir training

### Automatic Mixed Precision (AMP) + Gradient Accumulation

- **AMP (`torch.cuda.amp`)**: Komputasi dilakukan dalam `float16` untuk operasi forward/backward, namun bobot disimpan dalam `float32`. Ini memotong penggunaan VRAM ~40-50% tanpa kehilangan akurasi numerik yang signifikan
- **Gradient Accumulation (2 step)**: Gradien diakumulasi selama 2 batch sebelum satu step optimizer. Effective batch size menjadi 16 (8×2), memimik perilaku batch besar yang diketahui stabilisasi training transformer

### Model Checkpointing

Hanya model dengan `balanced_accuracy` terbaik pada validation set yang disimpan per fold. Model ini yang digunakan untuk inferensi test set, menghindari penggunaan model dari epoch overfitting.

### Probability Averaging untuk Inferensi

Test set diprediksi oleh semua 5 fold. Output softmax (probabilitas per kelas) dari setiap fold dirata-ratakan (`test_probs += fold_probs / N_SPLITS`). Final prediksi diambil dari `argmax` probabilitas rata-rata. Pendekatan ini lebih robust daripada voting mayoritas karena memanfaatkan confidence setiap model, bukan hanya keputusan biner.

---

### Referensi Teknis Pipeline Training

> **AdamW:** Loshchilov, I., & Hutter, F. (2019). *Decoupled Weight Decay Regularization*. ICLR 2019. [arXiv:1711.05101](https://arxiv.org/abs/1711.05101)
>
> **Linear Warmup Schedule (fine-tuning transformer):** Devlin, J., Chang, M. W., Lee, K., & Toutanova, K. (2019). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL 2019. [arXiv:1810.04805](https://arxiv.org/abs/1810.04805)
>
> **Automatic Mixed Precision (AMP):** Micikevicius, P., Narang, S., Alben, J., et al. (2018). *Mixed Precision Training*. ICLR 2018. [arXiv:1710.03740](https://arxiv.org/abs/1710.03740)


In [ ]:
def train_and_infer(df_train, df_test, label_encoder, model_name):
    num_classes = len(label_encoder.classes_)
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
    except Exception as e:
        print(f"Peringatan: Tokenizer tidak ditemukan di path ({e}). Mendownload dari HuggingFace...")
        tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
    
    # Class Weights Injection
    class_weights = compute_class_weight('balanced', classes=np.unique(df_train['label_encoded']), y=df_train['label_encoded'].values)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)
    
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_metrics = []
    
    oof_preds = np.zeros(len(df_train))
    oof_labels = np.zeros(len(df_train))
    test_probs = np.zeros((len(df_test), num_classes))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(df_train, df_train['label_encoded'])):
        print(f"\n{'='*40}\n>>> FOLD {fold + 1}/{N_SPLITS} <<<\n{'='*40}")
        
        train_data = df_train.iloc[train_idx].reset_index(drop=True)
        val_data = df_train.iloc[val_idx].reset_index(drop=True)
        
        train_dataset = TweetDataset(train_data['clean_text'], train_data['label_encoded'], tokenizer, MAX_LEN)
        val_dataset = TweetDataset(val_data['clean_text'], val_data['label_encoded'], tokenizer, MAX_LEN)
        
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
        
        model = XLMRobertaMultiDropout(model_name, num_labels=num_classes)
        model = model.to(device)
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
        total_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
        warmup_steps = int(0.1 * total_steps)
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
        
        best_b_acc = 0.0
        best_model_path = os.path.join(MODELS_DIR, f"fold_{fold+1}_roberta_best.pt")
        
        #  Kaggle P100 / T4x2 Sangat Cepat dengan AMP
        scaler = torch.cuda.amp.GradScaler()
        
        for epoch in range(EPOCHS):
            model.train()
            total_loss = 0
            optimizer.zero_grad()
            
            progress_bar = tqdm(train_loader, desc=f"Fold {fold+1} Epoch {epoch+1}/{EPOCHS} [Train]")
            for i, batch in enumerate(progress_bar):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                with torch.cuda.amp.autocast():
                    logits = model(input_ids, attention_mask)
                    loss = loss_fn(logits, labels)
                
                loss = loss / GRAD_ACCUM_STEPS
                scaler.scale(loss).backward()
                total_loss += loss.item() * GRAD_ACCUM_STEPS
                
                if (i + 1) % GRAD_ACCUM_STEPS == 0 or (i + 1) == len(train_loader):
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad()
                    
                progress_bar.set_postfix({'loss': f"{loss.item() * GRAD_ACCUM_STEPS:.4f}"})
            
            avg_train_loss = total_loss / len(train_loader)
            
            model.eval()
            val_preds, val_labels = [], []
            with torch.no_grad():
                for batch in tqdm(val_loader, desc=f"Fold {fold+1} Epoch {epoch+1}/{EPOCHS} [Val]  "):
                    input_ids = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)
                    labels = batch['labels'].to(device)
                    
                    logits = model(input_ids, attention_mask)
                    preds = torch.argmax(logits, dim=1).flatten()
                    
                    val_preds.extend(preds.cpu().numpy())
                    val_labels.extend(labels.cpu().numpy())
            
            b_acc = balanced_accuracy_score(val_labels, val_preds)
            print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val B-Acc: {b_acc:.4f}")
            
            if b_acc > best_b_acc:
                best_b_acc = b_acc
                best_val_preds = val_preds
                best_val_labels = val_labels
                torch.save(model.state_dict(), best_model_path)
                
        fold_metrics.append(best_b_acc)
        oof_preds[val_idx] = best_val_preds
        oof_labels[val_idx] = best_val_labels
        print(f"--> Fold {fold+1} Best Validation Balanced Accuracy: {best_b_acc:.4f}")
        
        print(f"Mengeksekusi prediksi Test Data menggunakan model Fold {fold+1}...")
        model.load_state_dict(torch.load(best_model_path, weights_only=True))
        model.eval()
        
        test_dataset = TweetDataset(df_test['clean_text'], None, tokenizer, MAX_LEN)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, shuffle=False)
        
        fold_test_probs = []
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Predicting Test Set (Fold {fold+1})"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                
                logits = model(input_ids, attention_mask)
                probs = torch.softmax(logits, dim=1)
                fold_test_probs.append(probs.cpu().numpy())
                
        fold_test_probs = np.vstack(fold_test_probs)
        test_probs += fold_test_probs / N_SPLITS
        
        # Free memory untuk fold selanjutnya
        del model
        gc.collect()
        torch.cuda.empty_cache()
        
    print(f"\n==============================================")
    print(f"🌟 FINAL CV SCORE XLM-ROBERTA-LARGE: {np.mean(fold_metrics):.4f} (±{np.std(fold_metrics):.4f}) 🌟")
    print(f"==============================================\n")
    
    print("=== 📊 OUT-OF-FOLD (OOF) CONFUSION MATRIX ===")
    print(confusion_matrix(oof_labels, oof_preds))
    print("\n=== 📈 OUT-OF-FOLD CLASSIFICATION REPORT ===")
    print(classification_report(oof_labels, oof_preds, target_names=label_encoder.classes_))
    
    print("Menghasilkan file submission.csv berdasarkan Template Resmi Kaggle...")
    final_preds = np.argmax(test_probs, axis=1)
    
    submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    df_test['pred_label'] = label_encoder.inverse_transform(final_preds)
    
    id_col = submission.columns[0]
    label_col = submission.columns[1]
    
    submission[label_col] = df_test['pred_label'].values
    
    submission_path = "/kaggle/working/submission_roberta.csv"
    submission.to_csv(submission_path, index=False)
    print(f"✅ Submission berhasil disimpan di: {submission_path}")

def run_pipeline():
    df_train, df_test, label_encoder = load_data()
    if df_train is not None:
        train_and_infer(df_train, df_test, label_encoder, MODEL_NAME)

if __name__ == "__main__":
    run_pipeline()

In [ ]:
# Konversi submission CSV ke XLSX
csv_path = "/kaggle/working/submission_roberta.csv"
xlsx_path = "/kaggle/working/submission_roberta.xlsx"

try:
    df_sub = pd.read_csv(csv_path)
    df_sub.to_excel(xlsx_path, index=False, engine='openpyxl')
    print(f"✅ File submission berformat XLSX berhasil disimpan di: {xlsx_path}")
except Exception as e:
    print(f"❌ Gagal mengonversi file: {e}\nPastikan module 'openpyxl' sudah terinstall (pip install openpyxl).")

---

## Daftar Pustaka

1. Conneau, A., Khandelwal, K., Goyal, N., Chaudhary, V., Wenzek, G., Guzmán, F., Grave, E., Ott, M., Zettlemoyer, L., & Stoyanov, V. (2020). Unsupervised Cross-lingual Representation Learning at Scale. *Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics (ACL 2020)*. https://arxiv.org/abs/1911.02116

2. Liu, Y., Ott, M., Goyal, N., Du, J., Joshi, M., Chen, D., Levy, O., Lewis, M., Zettlemoyer, L., & Stoyanov, V. (2019). RoBERTa: A Robustly Optimized BERT Pretraining Approach. *arXiv preprint*. https://arxiv.org/abs/1907.11692

3. Inoue, H. (2019). Multi-Sample Dropout for Accelerated Training and Better Generalization. *arXiv preprint*. https://arxiv.org/abs/1905.09788

4. Loshchilov, I., & Hutter, F. (2019). Decoupled Weight Decay Regularization. *7th International Conference on Learning Representations (ICLR 2019)*. https://arxiv.org/abs/1711.05101

5. Devlin, J., Chang, M. W., Lee, K., & Toutanova, K. (2019). BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding. *Proceedings of NAACL 2019*. https://arxiv.org/abs/1810.04805

6. Micikevicius, P., Narang, S., Alben, J., Diamos, G., Elsen, E., Garcia, D., Ginsburg, B., Houston, M., Kuchaiev, O., Venkatesh, G., & Wu, H. (2018). Mixed Precision Training. *6th International Conference on Learning Representations (ICLR 2018)*. https://arxiv.org/abs/1710.03740
